# Bubble evolution within one train

Follows **every frame of one train** (248 frames, 886 ns apart ≈ 220 µs) and produces the same analyses as
`bubble_analysis.ipynb`, as a function of time. It also makes a GIF of the area-coloured bubbles.

Set `TRAIN` in CONFIG. Everything for that train goes to `results/trains/tid<TRAIN>/`:

| file | content |
|---|---|
| `shapes/` | frame + bubble outlines per frame (editable in napari, like any inference run) |
| `bubbles.csv`, `frames.csv` | measurements (written by section 1) |
| `frame_metrics.csv` | per-frame metrics (section 3) |
| `figures/` | all plots and `overlays_area.gif` |

**Section 1 predicts all frames with your model** (frames that already have a draft are not predicted again) and exports
them. Reviewing 248 frames by hand is not practical, so unreviewed frames are included by default
(`INCLUDE_UNREVIEWED`). Any frames you do correct in napari (`python annotate_bubbles.py results/trains/tid3/shapes`)
are used automatically when you re-run section 1.

Within a train, frame-to-frame changes are partly measurement noise (the model is not identical on two almost identical
frames). The line plots therefore also show a running mean (`SMOOTH` frames).

In [ ]:
import os, time, dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import bubble_rcnn as br
import bubble_io as bio
import bubble_stats as bst

bst.style()

## CONFIG

In [ ]:
TRAIN = 3
RUN_DIR = os.path.join("results", "trains", f"tid{TRAIN}")
FRAME_GLOB = "raw/jpg_r563_svd_normalised_tr050/*.jpg"
MODEL_PATH = "models/bubble_rcnn_best.pt"

# --- section 1: prediction + export ---
RUN_PREDICTION = True        # predict frames that have no draft yet (needs the GPU kernel)
RUN_EXPORT = True            # (re)measure all frames -> bubbles.csv / frames.csv
INCLUDE_UNREVIEWED = True    # use frames you have not ticked "Reviewed"
DRAFT_KIND = "polygon"
INFER = dict(                # same meaning as in bubble_inference.ipynb
    score_thresh=0.5, mask_thresh=0.5, mask_nms_iou=0.7, um_per_px=3.2,
    edge_min_arc_deg=100.0, edge_ellipse_gain=1.6, edge_max_axis_ratio=2.5, edge_reliable_arc_deg=140.0,
    drop_unreliable_fits=False, drop_edge_bubbles=False, inner_margin=0.0,
)

# --- analysis ---
FRAME_INTERVAL_US = 0.886    # HPV-X2 framing time (886 ns)
TIME_LABEL = "time [µs]"
UM_PER_PX = 3.2
SMOOTH = 5                   # running mean over this many frames in the line plots (1 = off)
N_SNAPSHOTS = 8              # frames shown in histograms / overlay grid / density maps (evenly spaced)
COMPUTE_COVERAGE = True      # union of outlines per frame (~1 min for 248 frames)
HIST_VAR, HIST_LABEL = "r_eq_um", "equivalent radius [µm]"
SIZE_CLASSES_UM = [0, 10, 20, 40, np.inf]
OVERLAY_CMAP = "RdYlGn_r"    # green (small) -> red (large); "viridis" is colour-blind safe
OVERLAY_ALPHA = 0.45
DENSITY_CELL_UM = 80

# --- GIF ---
MAKE_GIF = True
GIF_EVERY = 1                # use every n-th frame (2 = half the frames, half the file size)
GIF_FPS = 12
GIF_DPI = 80                 # resolution of the GIF frames

SAVE_FIGS = True
FIG_DIR = os.path.join(RUN_DIR, "figures")

def save(fig, name):
    if SAVE_FIGS:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight")

## 1. Predict and export all frames of the train

In [ ]:
mcfg = dataclasses.replace(br.RCNNConfig.from_json(MODEL_PATH.replace(".pt", "_config.json")), **INFER)

if RUN_PREDICTION:
    import torch
    print("CUDA:", torch.cuda.is_available())
    index = bio.index_frames(FRAME_GLOB)
    paths = bio.select_frames([(TRAIN, "all")], index)
    model, _ = br.load_model(MODEL_PATH)
    bio.write_run_info(RUN_DIR, MODEL_PATH, mcfg, extra=dict(train=TRAIN))
    t0 = time.time()
    summary = bio.draft_frames(model, mcfg, paths, RUN_DIR, model_path=MODEL_PATH, kind=DRAFT_KIND, show_every=25)
    print(f"prediction done in {time.time() - t0:.0f}s:", summary["status"].value_counts().to_dict())

if RUN_EXPORT:
    t0 = time.time()
    bio.export_results(RUN_DIR, mcfg, include_unreviewed=INCLUDE_UNREVIEWED)
    print(f"export done in {time.time() - t0:.0f}s")

## 2. Load

In [ ]:
bubbles, frames = bio.load_results(RUN_DIR, reviewed_only=not INCLUDE_UNREVIEWED, in_analysis_only=True)
bubbles["t"] = bubbles["frame_idx"] * FRAME_INTERVAL_US
frames["t"] = frames["frame_idx"] * FRAME_INTERVAL_US
frames = frames.sort_values("t").reset_index(drop=True)
NATIVE_HW = (int(frames["image_height_px"].iloc[0]), int(frames["image_width_px"].iloc[0]))
print(f"train {TRAIN}: {len(frames)} frames ({frames['reviewed'].sum()} reviewed), {len(bubbles)} bubbles, "
      f"t = {frames['t'].min():.1f} … {frames['t'].max():.1f} µs")
frame_label = lambda r: f"{r['t']:.1f} µs (frame {int(r['frame_idx'])})"

## 3. Metrics per frame

In [ ]:
t0 = time.time()
m = bst.frame_metrics(bubbles, frames, run_dir=RUN_DIR if COMPUTE_COVERAGE else None, um_per_px=UM_PER_PX)
m.to_csv(os.path.join(RUN_DIR, "frame_metrics.csv"), index=False)
snap = bst.select_snapshots(m, N_SNAPSHOTS)
print(f"metrics for {len(m)} frames in {time.time() - t0:.0f}s; snapshots at frames {snap['frame_idx'].tolist()}")
cols = ["frame_idx", "t", "n_bubbles", "density_per_mm2", "r_mean_um", "r_median_um", "r32_um", "r_max_um",
        "polydispersity", "area_total_um2", "volume_um3", "frac_noncircular"] + (["coverage_frac"] if COMPUTE_COVERAGE else [])
display(m[cols].describe().loc[["mean", "std", "min", "max"]].round(3))

## 4. Bubble count

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
bst.plot_count(m, TIME_LABEL, ax=ax, smooth=SMOOTH)
save(fig, "count")

## 5. Average bubble size

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
bst.plot_mean_size(m, TIME_LABEL, ax=ax, smooth=SMOOTH)
save(fig, "mean_size")

## 6. Size distributions

The heatmap shows every frame's size distribution as one column. A band drifting upwards means growth, and
fading low bands mean small bubbles disappearing. Below it are histograms, cumulative and scaled distributions for
`N_SNAPSHOTS` evenly spaced frames.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2))
bst.plot_size_heatmap(bubbles, m, TIME_LABEL, var=HIST_VAR, label=HIST_LABEL, ax=ax)
save(fig, "size_heatmap")

In [ ]:
fig = bst.plot_histograms(bubbles, snap, var=HIST_VAR, xlabel=HIST_LABEL, frame_label=frame_label)
save(fig, "histograms")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
bst.plot_cdfs(bubbles, snap, var=HIST_VAR, xlabel=HIST_LABEL, ax=ax[0], frame_label=frame_label)
bst.plot_scaled_distribution(bubbles, snap, ax=ax[1], frame_label=frame_label)
fig.tight_layout()
save(fig, "distributions")

## 7. Size classes and coalescence check

In [ ]:
counts, labels = bst.size_class_counts(bubbles, SIZE_CLASSES_UM)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
bst.plot_size_classes(counts, labels, TIME_LABEL, ax=ax[0], smooth=SMOOTH)
bst.plot_relative(m, TIME_LABEL, ax=ax[1], smooth=SMOOTH)
fig.tight_layout()
save(fig, "size_classes_relative")

## 8. Overlays coloured by bubble area + GIF

One shared logarithmic colour scale for all frames (green = small, red = large). The GIF contains every
`GIF_EVERY`-th frame. With 248 frames it takes a few minutes and is roughly 10–30 MB; use `GIF_EVERY = 2` or a lower
`GIF_DPI` to make it smaller.

In [ ]:
norm = bst.area_norm(bubbles, "area_um2")
ncols = 2
nrows = int(np.ceil(len(snap) / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5.2 * nrows), squeeze=False)
for ax, (_, r) in zip(axs.ravel(), snap.iterrows()):
    img, shapes = bio.load_frame(RUN_DIR, r["frame"])
    b = bubbles[bubbles["frame"] == r["frame"]]
    bst.plot_area_overlay(img, [shapes[i] for i in b["bubble_id"]], b["area_um2"].to_numpy(), norm,
                          cmap=OVERLAY_CMAP, ax=ax, alpha=OVERLAY_ALPHA, title=f"{frame_label(r)}: {len(b)} bubbles")
for ax in axs.ravel()[len(snap):]:
    ax.set_visible(False)
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=OVERLAY_CMAP), ax=axs.ravel().tolist(), shrink=0.6,
             label="bubble area [µm²]")
save(fig, "overlays_area")

In [ ]:
if MAKE_GIF:
    t0 = time.time()
    gif_path = os.path.join(FIG_DIR, "overlays_area.gif")
    path, n = bst.make_overlay_gif(RUN_DIR, bubbles, m, norm, gif_path, cmap=OVERLAY_CMAP, alpha=OVERLAY_ALPHA,
                                   every=GIF_EVERY, fps=GIF_FPS, dpi=GIF_DPI,
                                   time_fmt=lambda r: f"train {TRAIN}  ·  {frame_label(r)}")
    print(f"GIF with {n} frames: {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {time.time() - t0:.0f}s)")

## 9. Spatial distribution and drift

In [ ]:
fig = bst.plot_density_maps(bubbles, snap, NATIVE_HW, um_per_px=UM_PER_PX, cell_um=DENSITY_CELL_UM, frame_label=frame_label)
save(fig, "density_maps")
img0, _ = bio.load_frame(RUN_DIR, m["frame"].iloc[0])
fig = bst.plot_drift(m, NATIVE_HW, um_per_px=UM_PER_PX, img=img0, time_label=TIME_LABEL)
save(fig, "drift")

## 10. Shape

In [ ]:
fig = bst.plot_shape(bubbles, m, TIME_LABEL, smooth=SMOOTH)
save(fig, "shape")

## Notes

* Frames are 886 ns apart, so changes within one train are small. The running mean separates real trends from
  frame-to-frame scatter of the model. The standard deviation in section 3 gives the size of that scatter.
* Unreviewed frames are raw model output. Spot-check a few in napari, especially if a plot shows sudden jumps.
* To compare trains, run this notebook with another `TRAIN`. Each train gets its own folder.